Цель проекта: Определить мутации, ответственные за устойчивость к ампициллину в секвенированном штамме E. coli, и предложить механизмы резистентности, а также альтернативные антибиотики для лечения.

In [1]:
# создаем директорию и папку для упорядочивания ведения проекта
mkdir -p ~/project_ampR/raw_data
cd ~/project_ampR

SyntaxError: invalid syntax (1711381086.py, line 1)

In [ ]:
# Шаг №2: загружаем данные
# закгрузка референсного генома E. coli K-12 MG1655 через команду wget
cd raw_data
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gff.gz
gunzip *.gz


In [ ]:
# скачиваем сырые риды
# не получилось сдлеать это по ссылке, тк она не открывалось, пришлось скачивать с диска через установку на свой компьютер
cp /mnt/c/Users/ipols/Downloads/amp_res_1.fastq/amp_res_1.fastq ~/project_ampR/raw_data/
cp /mnt/c/Users/ipols/Downloads/amp_res_2.fastq/amp_res_2.fastq ~/project_ampR/raw_data/
cp /mnt/c/Users/ipols/Downloads/amp_res_1.fastq/raw.fastq ~/project_ampR/raw_data/amp_res_1.fastq
cp /mnt/c/Users/ipols/Downloads/amp_res_2.fastq/raw.fastq ~/project_ampR/raw_data/amp_res_2.fastq
ls -lh ~/project_ampR/raw_data/ # проверка


In [ ]:
# Шаг №3: начало работы
# подсчет количества ридов в каждом файле
zcat amp_res_1.fastq.gz | wc -l
1823504

In [ ]:
# смотрим статистику
conda install -c bioconda seqkit
seqkit stats amp_res_1.fastq.gz amp_res_2.fastq.gz

# результат вывода: Длина ридов 101 bp, парные
 avg_len  max_len
amp_res_1.fastq.gz  FASTQ   DNA    455,876  46,043,476      101      101      101
amp_res_2.fastq.gz  FASTQ   DNA    455,876  46,043,476      101      101      101

In [ ]:
# Шаг №4: запуск fastqc на сырых данных
fastqc -o . amp_res_1.fastq.gz amp_res_2.fastq.gz
# необходимо обрезать адаптеры и отфильтровать по качеству

In [ ]:
#Шаг №5: фильтрация с помощью Trimmomatic
conda install -c bioconda trimmomatic
# запуск команды
trimmomatic PE \
    amp_res_1.fastq.gz amp_res_2.fastq.gz \
    amp_res_1_trimmed_P.fq amp_res_1_trimmed_U.fq \
    amp_res_2_trimmed_P.fq amp_res_2_trimmed_U.fq \
    LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20

# результат работы команды
fq LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20
Quality encoding detected as phred33
Input Read Pairs: 455876
Both Surviving: 446259 (97.89%)
Forward Only Surviving: 9216 (2.02%)
Reverse Only Surviving: 273 (0.06%)
Dropped: 128 (0.03%)
TrimmomaticPE: Completed successfully

# число ридов после обрезки
 wc -l amp_res_1_trimmed_P.fq
# 1785036 amp_res_1_trimmed_P.fq

In [ ]:
# Шаг №6: повторный FastQC после обрезки
fastqc -o . amp_res_1_trimmed_P.fq amp_res_2_trimmed_P.fq

In [ ]:
# Шаг 7: выравнивание на референс (BWA-MEM)
conda install -c bioconda bwa
bwa index GCF_000005845.2_ASM584v2_genomic.fna #создаем файлы .amb, .ann, .bwt, .pac, .sa

# выравниваем парные риды через команду
bwa mem GCF_000005845.2_ASM584v2_genomic.fna \
    amp_res_1_trimmed_P.fq amp_res_2_trimmed_P.fq > alignment.sam

In [ ]:
# сортируем и индексируем
samtools view -S -b alignment.sam > alignment.bam
samtools sort alignment.bam -o alignment_sorted.bam
samtools index alignment_sorted.bam

In [ ]:
# смотрим статистику выравнивания
samtools flagstat alignment_sorted.bam
# получили отличный результат - 99.56%

In [ ]:
# Шаг №8: фильтрация данных  с помощью VarScan
# создаем mpileup
samtools mpileup -f GCF_000005845.2_ASM584v2_genomic.fna \
    alignment_sorted.bam > my.mpileup

# устанавливаем VarScan и вызываем SNP
conda install -c bioconda varscan
varscan mpileup2snp my.mpileup --min-var-freq 0.8 --variants --output-vcf 1 > VarScan_results.vcf
# частота 80% была выюрана, чтобы избежать дожных вызовов из-за ошибок в секверировании

In [ ]:
# результат
Min coverage:   8
Min reads2:     2
Min var freq:   0.8
Min avg qual:   15
P-value thresh: 0.01
Reading input from my.mpileup
4641343 bases in pileup file
9 variant positions (6 SNP, 3 indel)
0 were failed by the strand-filter
6 variant positions reported (6 SNP, 0 indel)

In [ ]:
# открываем файл и вмдим шесть мутаций (chr, позиция, референсный аллель, альтернативный, частота)
cat VarScan_results.vcf

In [ ]:
# Шаг №9: долго долго мучаю с IGV и делаю всё по инструкции, чтобы получить две полоски на белом фоне...

In [ ]:
# Шаг №10:  Автоматическая аннотация SNP (SnpEff)
cd ~/project_ampR
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gbff.gz
gunzip GCF_000005845.2_ASM584v2_genomic.gbff.gz # скачаваем genbank
conda install -c bioconda snpeff # установка
snpEff build -genbank -v k12 -c snpEff.config # сборка базы
snpEff ann -c snpEff.config k12 VarScan_results.vcf > VarScan_results_annotated.vcf #аннотация
cat VarScan_results_annotated.vcf | grep -v "^#" | cut -f 1-8 # смотрим результаты

In [ ]:
#Шаг № 11: снова мучаюсь с IGV